# 00 · Descarga y consolidación de datos

Construye los dos datasets base del proyecto:

- **Retornos diarios reales** (hasta ~36 años, Norgate) para el universo
  reducido (25 bancos, backbone del predictor final) y el universo amplio
  (150 bancos, usado solo para entrenar los generadores con más datos).
- **Barras de 5 minutos reales** (EODHD, desde 2020-11, todo lo que sirve
  el API, ~5,5 años) y sus features
  intradía derivadas (volatilidad realizada, retorno de apertura/cierre,
  rango), para el universo amplio.

Todo se cachea en `datos/raw/` y `datos/interim/` (gitignored, pesan
demasiado y contienen la ruta al `APkey`); este notebook solo hay que
relanzarlo si cambia el universo, las fechas, o se borra la caché.

**Por qué dos universos distintos** (`src/config.py`): el predictor final
necesita ~30 años de retorno diario REAL por banco (solo 25 bancos los
tienen completos en el dump de Norgate). Los generadores, en cambio, solo
necesitan datos de la ventana real (2020-11 en adelante), así que para
darles más muestras
con las que aprender bien la distribución conjunta (retorno, features
intradía) se usan hasta 150 bancos, aunque no coticen desde 1990.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

# Recarga automatica de src/ al editarlo: sin esto, si se edita un modulo
# de src/ con el kernel ya arrancado, Jupyter sigue usando la version que
# importo la primera vez (y aparecen errores tipo "unexpected keyword
# argument" con codigo que en disco si es correcto).
try:
    ip = get_ipython()
    ip.run_line_magic("load_ext", "autoreload")
    ip.run_line_magic("autoreload", "2")
except NameError:
    pass  # ejecutandose fuera de IPython/Jupyter

import numpy as np
import pandas as pd

from src import config, data_eodhd as de, data_norgate as dn, features as feat

## 1. Retornos diarios reales (Norgate)

Universo reducido (25 bancos, ~36 años completos).

In [2]:
prices_predictor = dn.load_daily_prices(config.PREDICTOR_TICKERS)
returns_predictor = dn.compute_log_returns(prices_predictor)
print("precios:", prices_predictor.shape, " retornos:", returns_predictor.shape)

cobertura = dn.coverage_report(prices_predictor)
cobertura

precios: (9169, 25)  retornos: (7865, 25)


,first_date,last_date,n_obs,pct_nan
ticker,,,,
BAC,1990-01-02,2026-05-29,9169,0.000000
WFC,1990-01-02,2026-05-29,9169,0.000000
JPM,1990-01-02,2026-05-29,9169,0.000000
C,1990-01-02,2026-05-29,9169,0.000000
HBAN,1990-01-02,2026-05-29,9169,0.000000
USB,1990-01-02,2026-05-29,9169,0.000000
TFC,1990-01-02,2026-05-29,9168,0.000109
KEY,1990-01-02,2026-05-29,9169,0.000000
RF,1990-01-02,2026-05-29,9169,0.000000


In [3]:
assert cobertura["pct_nan"].max() < 0.05, "Algún ticker del universo predictor tiene demasiados huecos"
cobertura.to_csv(config.TABLES_DIR / "00_cobertura_predictor.csv")

Universo amplio (150 bancos), retorno diario real solo desde el inicio de
la ventana real (2020-11 en adelante, es lo único que necesitan los
generadores).

In [4]:
prices_generator = dn.load_daily_prices(
    config.GENERATOR_TICKERS, start=config.REAL_INTRADAY_START_DATE
)
# dropna=None: NO exigimos que los 150 bancos coticen el mismo dia (varios
# no tienen historia completa a proposito, ver seccion introductoria). Cada
# ticker conserva sus propios NaN; build_conditional_pool los filtra 1 a 1.
returns_generator = dn.compute_log_returns(prices_generator, dropna=None)
print("precios:", prices_generator.shape, " retornos:", returns_generator.shape)
print(f"NaNs: {returns_generator.isna().mean().mean():.1%} de media por ticker (normal: altas/bajas parciales)")

precios: (1399, 150)  retornos: (1399, 150)
NaNs: 5.3% de media por ticker (normal: altas/bajas parciales)


## 2. Barras de 5 minutos reales (EODHD)

Se descargan (o se leen de caché) para el universo amplio. La API key se
lee de `datos/APkey` (gitignored) y nunca se imprime.

In [5]:
bars_by_ticker = de.download_universe_5m(tickers=config.GENERATOR_TICKERS)

BAC     114925 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
WFC     114927 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
JPM     114928 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
C       114928 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
HBAN    114926 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
USB     114922 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
TFC     114928 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
KEY     114925 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
RF      114926 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
FITB    114929 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
VLY     114928 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
FHN     114929 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
CFG     114917 barras  [2020-11-02 14:30

WAL     114927 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
CLBK    114684 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
FCF     114922 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
FIBK    114926 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
SSB     114915 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
FFBC    114886 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
SBCF    114908 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
WSBC    114903 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
RNST    114913 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
PB      114921 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
BRBS    112163 barras  [2020-12-16 14:45:00+00:00 -> 2026-08-28 20:00:00+00:00]
BUSE    114870 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
BBT      19635 barras  [2025-09-02 13:30

BY      114779 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
NBBK     52457 barras  [2023-12-28 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
TCBI    114863 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
TBBK    114900 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
DCOM    114726 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
NFBK    114179 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-19 19:55:00+00:00]
FBNC    114756 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
BOH     114903 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
AMTB    114778 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
STBA    114676 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
NBHC    114874 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
FMNB    114686 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
GABC    114562 barras  [2020-11-02 14:30

PDLB    114402 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
TFIN     73624 barras  [2022-12-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
MPB     114617 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
CASH    114761 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
NRIM    114667 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
CARE    114616 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
MSBI    114740 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
FSBC    104756 barras  [2021-05-05 13:40:00+00:00 -> 2026-08-28 20:00:00+00:00]
CBAN    114533 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
RVSB    114382 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
CIVB    114702 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
IBCP    114661 barras  [2020-11-02 14:30:00+00:00 -> 2026-08-28 20:00:00+00:00]
WNEB    114381 barras  [2020-11-02 14:30

In [6]:
n_ok = sum(1 for df in bars_by_ticker.values() if len(df) > 0)
print(f"{n_ok}/{len(bars_by_ticker)} tickers con barras de 5 min descargadas")

cobertura_intradia = pd.DataFrame(
    {
        "ticker": list(bars_by_ticker.keys()),
        "n_bars": [len(df) for df in bars_by_ticker.values()],
        "first": [df.index.min() if len(df) else pd.NaT for df in bars_by_ticker.values()],
        "last": [df.index.max() if len(df) else pd.NaT for df in bars_by_ticker.values()],
    }
).set_index("ticker")
cobertura_intradia.to_csv(config.TABLES_DIR / "00_cobertura_intradia.csv")
cobertura_intradia.sort_values("n_bars").head(10)

150/150 tickers con barras de 5 min descargadas


,n_bars,first,last
ticker,,,
SICP,392,2026-06-18 20:00:00+00:00,2026-08-28 19:55:00+00:00
BBT,19635,2025-09-02 13:30:00+00:00,2026-08-28 20:00:00+00:00
NPB,30125,2025-02-13 21:00:00+00:00,2026-08-28 20:00:00+00:00
FBLA,35887,2024-10-23 14:45:00+00:00,2026-08-28 20:00:00+00:00
FLG,36145,2024-10-28 13:30:00+00:00,2026-08-28 20:00:00+00:00
UCB,40719,2024-08-06 13:30:00+00:00,2026-08-28 20:00:00+00:00
NBBK,52457,2023-12-28 14:30:00+00:00,2026-08-28 20:00:00+00:00
OBK,64365,2023-05-22 13:30:00+00:00,2026-08-28 20:00:00+00:00
MCHB,71590,2021-01-14 16:40:00+00:00,2026-08-28 20:00:00+00:00


## 3. Features intradía diarias (volatilidad realizada, etc.)

Primero se descarta, DÍA A DÍA, cualquier sesión con menos de
`MIN_BARS_PER_SESSION` barras (feed caído, apertura tardía — no cierres
anticipados legítimos por festivo, esos sí se quedan). Después, un ticker
entra en el pool de entrenamiento de los generadores solo si le quedan al
menos `MIN_SESSIONS_FOR_GENERATOR_POOL` sesiones válidas (filtra bancos
intervenidos/fusionados a mitad de la ventana real).

In [7]:
intraday_feats_all = {
    tk: feat.daily_intraday_features(bars) for tk, bars in bars_by_ticker.items()
}
intraday_feats_all = {
    tk: f[f["n_bars"] >= config.MIN_BARS_PER_SESSION] for tk, f in intraday_feats_all.items()
}
intraday_feats = {
    tk: f for tk, f in intraday_feats_all.items()
    if len(f) >= config.MIN_SESSIONS_FOR_GENERATOR_POOL
}
print(
    f"{len(intraday_feats)}/{len(intraday_feats_all)} tickers con "
    f">= {config.MIN_SESSIONS_FOR_GENERATOR_POOL} sesiones intradía válidas"
)

149/150 tickers con >= 60 sesiones intradía válidas


## 4. Pool condicional (retorno diario real, features intradía reales)

Dataset de entrenamiento de los 4 generadores del notebook 02: cada fila
es un día real de un banco cualquiera del universo amplio, con su retorno
diario y sus 4 features intradía reales. Cuantas más muestras, mejor
generalizan los generadores — de ahí usar el universo amplio.

In [8]:
pool, pool_meta = feat.build_conditional_pool(returns_generator, intraday_feats)
print("pool:", pool.shape, "  columnas:", ["log_return", *feat.INTRADAY_FEATURE_COLS])
pool_meta["ticker"].value_counts().describe()

pool: (141065, 5)   columnas: ['log_return', 'realized_vol', 'open_30m_ret', 'close_30m_ret', 'hl_range']


count     149.000000
mean      946.744966
std       444.189030
min        21.000000
25%       587.000000
50%      1078.000000
75%      1355.000000
max      1398.000000
Name: count, dtype: float64

In [9]:
assert len(pool) > 5000, "Pool de entrenamiento de los generadores demasiado pequeño"

## 5. Guardar datasets intermedios (`datos/interim/`, gitignored)

In [10]:
np.save(config.INTERIM_DIR / "conditional_pool.npy", pool)
pool_meta.to_parquet(config.INTERIM_DIR / "conditional_pool_meta.parquet")
returns_predictor.to_parquet(config.INTERIM_DIR / "returns_predictor.parquet")
returns_generator.to_parquet(config.INTERIM_DIR / "returns_generator.parquet")

intraday_long = (
    pd.concat([f.assign(ticker=tk) for tk, f in intraday_feats.items()])
    .rename_axis("date")
    .reset_index()
)
intraday_long.to_parquet(config.INTERIM_DIR / "intraday_features_real.parquet")

print("Guardado en", config.INTERIM_DIR)

Guardado en /Users/emiliosanchez/TallerB5_T1/Taller-B5-T1-Generativos/datos/interim
